In [91]:
print("Hello World")

Hello World


## Setup

In [120]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost
from sklearn.metrics import fbeta_score
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import numpy as np
import os

In [93]:
print(torch.__file__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/home/ub089/.conda/envs/venv/lib/python3.10/site-packages/torch/__init__.py
2.6.0+cu124
12.4
NVIDIA A100-SXM4-40GB
2
cuda


In [94]:
SAMPLE_SUBMISSION_CSV = "./sleep/sample_submission.csv"
TRAIN_DIR = "./sleep/train/train/"
TEST_DIR = "./sleep/test_segment/test_segment/"

In [95]:
test_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
test_df

,id,labels
0,test001_00000,W
1,test001_00001,W
2,test001_00002,W
3,test001_00003,NaN
4,test001_00004,NaN
...,...,...
7827,test010_00754,NaN
7828,test010_00755,NaN
7829,test010_00756,NaN
7830,test010_00757,NaN


In [96]:
pd.read_csv(TEST_DIR + test_df.iloc[0]['id'][0:7] + '/' + test_df.iloc[0]['id'] + '.csv')

,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI
0,-16.787332,-29.654620,-35.583667,44.486126,30.773655,0.769030,52.986465,0.911322
1,2.375216,-30.549130,-35.593414,44.482602,30.773662,0.769137,52.986463,0.911322
2,46.691694,-29.650487,-35.582574,44.487709,30.773652,0.768899,52.986385,0.911322
3,126.648750,-29.604712,-35.591203,44.482427,30.773663,0.769318,52.986546,0.911322
4,208.143510,-30.267480,-35.588414,44.486243,30.773655,0.767400,52.986326,0.911322
...,...,...,...,...,...,...,...,...
475,12.685158,-29.858239,-34.603998,45.472498,30.734090,0.772641,52.274134,1.127938
476,3.025203,-30.528547,-34.592616,45.477440,30.734173,0.774426,52.275429,1.127244
477,-14.935005,-29.557710,-34.606103,45.467243,30.734058,0.775582,52.274324,1.127918
478,-37.082422,-29.920104,-34.594194,45.486995,30.734153,0.775335,52.274654,1.127482


In [97]:
train_df = pd.DataFrame([{}])
for index, item in enumerate(os.listdir(TRAIN_DIR)):
	now_df = pd.read_csv(TRAIN_DIR + item)
	train_df = pd.concat([train_df, now_df])
	if index == 10:
		break
train_df = train_df[1:387361]
train_df

,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage
0,-23.336159,-29.746921,-30.648611,48.422386,30.793429,0.578981,69.860958,0.787578,W
1,-34.456906,-29.973617,-31.536269,47.672690,30.793429,0.579145,69.861003,0.788018,W
2,-35.591778,-30.381488,-30.543433,46.540243,30.793429,0.578601,69.861263,0.787457,W
3,-18.914839,-30.744572,-29.517339,47.423375,30.793429,0.579992,69.860670,0.787991,W
4,-1.916808,-30.403062,-30.435433,47.515878,30.793429,0.570414,69.861516,0.787678,W
...,...,...,...,...,...,...,...,...,...
387355,-82.959629,13.805326,40.972485,45.325311,34.441233,0.565022,48.884007,1.189947,W
387356,-110.178548,13.849557,40.413724,45.572677,34.441071,0.564994,48.883827,1.188711,W
387357,-126.475464,13.850960,40.978387,45.405486,34.441332,0.564983,48.884089,1.177100,W
387358,-84.448504,13.816877,41.076331,45.511786,34.441071,0.565014,48.883839,1.172426,W


In [98]:
train_df['Sleep_Stage'].value_counts()

Sleep_Stage
N2    196320
W     120960
R      36480
N1     32160
N3      1440
Name: count, dtype: int64

In [99]:
train_df['Sleep_Stage'] = train_df['Sleep_Stage'].map({
	'N2': 0,
	'W': 1,
	'R': 2,
	'N1': 3,
	'N3': 4,
})

In [103]:
train_df

,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage
0,-23.336159,-29.746921,-30.648611,48.422386,30.793429,0.578981,69.860958,0.787578,1
1,-34.456906,-29.973617,-31.536269,47.672690,30.793429,0.579145,69.861003,0.788018,1
2,-35.591778,-30.381488,-30.543433,46.540243,30.793429,0.578601,69.861263,0.787457,1
3,-18.914839,-30.744572,-29.517339,47.423375,30.793429,0.579992,69.860670,0.787991,1
4,-1.916808,-30.403062,-30.435433,47.515878,30.793429,0.570414,69.861516,0.787678,1
...,...,...,...,...,...,...,...,...,...
387355,-82.959629,13.805326,40.972485,45.325311,34.441233,0.565022,48.884007,1.189947,1
387356,-110.178548,13.849557,40.413724,45.572677,34.441071,0.564994,48.883827,1.188711,1
387357,-126.475464,13.850960,40.978387,45.405486,34.441332,0.564983,48.884089,1.177100,1
387358,-84.448504,13.816877,41.076331,45.511786,34.441071,0.565014,48.883839,1.172426,1


In [104]:
train_df['Sleep_Stage'].value_counts()

Sleep_Stage
0    196320
1    120960
2     36480
3     32160
4      1440
Name: count, dtype: int64

## Data

In [102]:
model = xgboost.XGBClassifier(
	n_estimators=1000,
	learning_rate=0.02,
	max_depth=5,
	subsample=0.8,
	colsample_bytree=0.8,
	random_state=42
)
model

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [107]:
X = train_df.drop(columns=['Sleep_Stage'])
y = train_df['Sleep_Stage']

In [124]:
X

,BVP,ACC_X,ACC_Y,ACC_Z,TEMP,EDA,HR,IBI
0,-23.336159,-29.746921,-30.648611,48.422386,30.793429,0.578981,69.860958,0.787578
1,-34.456906,-29.973617,-31.536269,47.672690,30.793429,0.579145,69.861003,0.788018
2,-35.591778,-30.381488,-30.543433,46.540243,30.793429,0.578601,69.861263,0.787457
3,-18.914839,-30.744572,-29.517339,47.423375,30.793429,0.579992,69.860670,0.787991
4,-1.916808,-30.403062,-30.435433,47.515878,30.793429,0.570414,69.861516,0.787678
...,...,...,...,...,...,...,...,...
387355,-82.959629,13.805326,40.972485,45.325311,34.441233,0.565022,48.884007,1.189947
387356,-110.178548,13.849557,40.413724,45.572677,34.441071,0.564994,48.883827,1.188711
387357,-126.475464,13.850960,40.978387,45.405486,34.441332,0.564983,48.884089,1.177100
387358,-84.448504,13.816877,41.076331,45.511786,34.441071,0.565014,48.883839,1.172426


In [109]:
y

0         1
1         1
2         1
3         1
4         1
         ..
387355    1
387356    1
387357    1
387358    1
387359    1
Name: Sleep_Stage, Length: 387360, dtype: int64

In [125]:
X_train_agg = X.groupby(np.arange(len(X)) // 480).mean()
y_train_agg = y.groupby(np.arange(len(y)) // 480).first()

In [127]:
model = xgboost.XGBClassifier(
	n_estimators=1000,
	learning_rate=0.05,
	max_depth=5,
	random_state=42,
	device=str(device)
)

In [128]:
model.fit(X_train_agg, y_train_agg, verbose=100)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [130]:
predictions = []

model.set_params(device="cpu")

predictions = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting Sleep Stages'):
	file_path = f"{TEST_DIR}{row['id'][0:7]}/{row['id']}.csv"
	batch = pd.read_csv(file_path)
	features = batch.mean().to_frame().T
	
	pred = model.predict(features)[0]
	predictions.append(pred)

test_df['labels'] = predictions

Predicting Sleep Stages: 100%|██████████| 7832/7832 [00:35<00:00, 222.16it/s]


In [133]:
test_df['labels'] = test_df['labels'].astype(str)
test_df

,id,labels
0,test001_00000,1
1,test001_00001,1
2,test001_00002,1
3,test001_00003,1
4,test001_00004,1
...,...,...
7827,test010_00754,1
7828,test010_00755,1
7829,test010_00756,1
7830,test010_00757,1


In [135]:
test_df['labels'] = test_df['labels'].map({
	'0': 'N2',
	'1': 'W',
	'2': 'R',
	'3': 'N1',
	'4': 'N3',
})

In [137]:
test_df['labels'].value_counts()

labels
W     5552
N2    1916
N1     255
R      109
Name: count, dtype: int64

In [139]:
test_df.to_csv("submission.csv", index=False)